In [ ]:
import csv
from pathlib import Path

In [ ]:
scheduled_path = Path("raw-zip-scheduled")
scheduled_files = sorted(scheduled_path.glob("*.zip"))
print(len(scheduled_files))
scheduled_files[:5]

In [ ]:
import re
from collections import defaultdict, namedtuple

ScheduledOutage = namedtuple(
    "ScheduledOutage",
    [
        "timestamp",
        "ptid",
        "equipment_name",
        "scheduled_out_datetime",
        "scheduled_in_datetime",
        "bus1",
        "bus2",
        "voltage",
    ],
)
Interval = namedtuple("Interval", ["start", "end"])

equipment_name_pattern = r"^([A-Za-z0-9._ -]{8})-([A-Za-z0-9._ -]{8})_(\d{2,3})_(.+)$"

In [ ]:
zip_path = scheduled_files[0]

In [ ]:
import io
from datetime import datetime
from zipfile import ZipFile


def parse_scheduled_outage(row):
    equipment_name = row["Equipment Name"]
    m = re.match(equipment_name_pattern, equipment_name)
    if m is None:
        return None
    return ScheduledOutage(
        timestamp=datetime.strptime(row["Timestamp"], "%m/%d/%Y %H:%M:%S"),
        ptid=int(row["PTID"]),
        equipment_name=row["Equipment Name"],
        scheduled_out_datetime=datetime.strptime(
            row["Scheduled Out Date/Time"],
            "%m/%d/%Y %H:%M:%S",
        ),
        scheduled_in_datetime=datetime.strptime(
            row["Scheduled In Date/Time"],
            "%m/%d/%Y %H:%M:%S",
        ),
        bus1=m.group(1),
        bus2=m.group(2),
        voltage=int(m.group(3)),
    )


def list_csvs(zip_path):
    with ZipFile(zip_path) as z:
        return [i for i in sorted(z.namelist()) if i.endswith(".csv")]


def read_csv_from_zip(zip_path, csv_name):
    with ZipFile(zip_path) as z:
        with z.open(csv_name) as f:
            data = f.read()
        csv_reader = csv.DictReader(io.StringIO(data.decode("utf-8")))
        # Only keep rows with line outages
        data = [parse_scheduled_outage(row) for row in csv_reader]
        data = [row for row in data if row is not None]
    return data


def summarize_data(data):
    """Implement processing method described in this paper titled "Transmission grid outage statistics extracted from a webpage logging outages in northeast america" """
    # group by ptid
    by_ptid = defaultdict(list)
    for outage in data:
        by_ptid[outage.ptid].append(outage)
    # save (in, out) dates for each ptid
    summary = defaultdict(set)
    for ptid, rows in by_ptid.items():
        for row in rows:
            key = (row.scheduled_in_datetime, row.scheduled_out_datetime)
            summary[ptid].add(key)
    return summary


# Example using an existing variable in the notebook:
zip_path = scheduled_files[0]
print(zip_path)
print("members:", list_csvs(zip_path))

# Read one file from the zip (text)
member = list_csvs(zip_path)[0]
data = read_csv_from_zip(zip_path, member)
print(len(data))
summary = summarize_data(data)
print(len(summary))

In [ ]:
from tqdm import tqdm

scheduled_outages = defaultdict(dict)
for zip_path in tqdm(scheduled_files):
    for member in list_csvs(zip_path):
        data = read_csv_from_zip(zip_path, member)
        summary = summarize_data(data)
        for ptid, in_out_set in summary.items():
            scheduled_outages[ptid].update(in_out_set)

In [ ]:
scheduled_outages.keys()

In [ ]:
actual_outages[25013]

In [ ]:
import pandas as pd

actual_outage_csv_path = Path("processed-scheduled-outages.csv")

# build a flat table from the nested defaultdict
rows = []
for ptid, partitions in actual_outages.items():
    for outage_dt, interval in partitions.items():
        rows.append(
            {
                "PTID": ptid,
                "Name": interval.start.equipment_name,
                "OutDatetime": outage_dt,
                "MinTimeStamp": interval.start.timestamp,
                "MaxTimeStamp": interval.end.timestamp,
                "Voltage": interval.start.voltage,
                "FirstBus": interval.start.bus1,
                "SecondBus": interval.start.bus2,
            }
        )

rows = sorted(rows, key=lambda x: x["OutDatetime"])
df = pd.DataFrame(rows)

# write out; use the existing json path with a .csv suffix
df.to_csv(actual_outage_csv_path, index=False)

print(f"saved {len(df)} rows to {actual_outage_csv_path}")